# Diagnostic 3: partial freeze, shared head (resnet18, real/correct images)

In [2]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [19]:
import json
import pandas as pd
import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline
from src.models.diagnostic_partial_freeze import PartiallyFrozenSharedHeadPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'resnet18'
IMG_FEAT_DIM = 512
IMG_PROJ_DIM = 4

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic'
EXP2_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_3_resnet'
SUMMARY_PATH = RESULTS_DIR / 'diagnostic3_partial_freeze_shared_head_resnet18_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])
print('Diagnostic 3 uses train + validation only; the held-out test set is not loaded.')

batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]
Diagnostic 3 uses train + validation only; the held-out test set is not loaded.


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [7]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {ENCODER_NAME} | seed={seed}\n")

    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)

    train.set_seed(seed)  # before constructing the trainable portion -- weight init draws from this
    model = PartiallyFrozenSharedHeadPredictor(base_model=base_model, img_feat_dim=IMG_FEAT_DIM,
                                                img_proj_dim=IMG_PROJ_DIM)

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'diagnostic3_partial_freeze_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])
    val_loss_hist = history['val_loss']
    best_epoch = int(np.argmin(val_loss_hist) + 1)

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'seed': seed, 'n_epochs_trained': n_epochs, 'best_epoch': best_epoch,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_loss_hist],
        'val_results': val_results,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'diagnostic3_partial_freeze_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  best epoch={best_epoch:>3}  "
          f"validation RMSE @ {HORIZONS} min = {val_rmse}")

Test results — resnet18 | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: PartiallyFrozenSharedHeadPredictor  params: 1,029,421


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  16/50  best epoch=  6  validation RMSE @ [15, 30, 60, 90, 120] min = [10.56, 16.59, 23.92, 28.35, 31.15]
Test results — resnet18 | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: PartiallyFrozenSharedHeadPredictor  params: 1,029,421


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  16/50  best epoch=  6  validation RMSE @ [15, 30, 60, 90, 120] min = [10.52, 16.62, 23.99, 28.49, 31.28]
Test results — resnet18 | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: PartiallyFrozenSharedHeadPredictor  params: 1,029,421


...

seed= 56  stopped at epoch  15/50  best epoch=  5  validation RMSE @ [15, 30, 60, 90, 120] min = [10.6, 16.63, 23.92, 28.39, 31.29]
Test results — resnet18 | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: PartiallyFrozenSharedHeadPredictor  params: 1,029,421


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  19/50  best epoch=  9  validation RMSE @ [15, 30, 60, 90, 120] min = [10.57, 16.69, 23.99, 28.45, 31.29]
Test results — resnet18 | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: PartiallyFrozenSharedHeadPredictor  params: 1,029,421


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  17/50  best epoch=  7  validation RMSE @ [15, 30, 60, 90, 120] min = [10.52, 16.54, 23.87, 28.32, 31.17]


In [17]:
best_epochs = [r['best_epoch'] for r in all_results]
stopping_epochs = [r['n_epochs_trained'] for r in all_results]
loss_gaps = [r['val_loss_history'][-1] - r['train_loss_history'][-1] for r in all_results]

per_horizon_rmse = {}
for h in HORIZONS:
    vals = [r['val_results'][h]['rmse'] for r in all_results]
    per_horizon_rmse[h] = {'mean': float(np.mean(vals)), 'sd': float(np.std(vals, ddof=1))}

print('=== Diagnostic 3: partial freeze, shared head (resnet18, real images) ===')
print(f"Best epoch: {np.mean(best_epochs):.2f} +/- {np.std(best_epochs, ddof=1):.2f}; "
      f"stopping epoch: {np.mean(stopping_epochs):.2f} +/- {np.std(stopping_epochs, ddof=1):.2f}")
print(f"Final (val - train) loss gap: {np.mean(loss_gaps):.6f} +/- {np.std(loss_gaps, ddof=1):.6f}")
for h in HORIZONS:
    s = per_horizon_rmse[h]
    print(f"{h:>3}-min:  RMSE = {s['mean']:.2f} +/- {s['sd']:.2f}")

with open(SUMMARY_PATH, 'w') as f:
    json.dump({
        'seeds': SEEDS, 'img_proj_dim': IMG_PROJ_DIM,
        'best_epoch': {'mean': float(np.mean(best_epochs)), 'sd': float(np.std(best_epochs, ddof=1))},
        'stopping_epoch': {'mean': float(np.mean(stopping_epochs)),
                            'sd': float(np.std(stopping_epochs, ddof=1))},
        'final_loss_gap': {'mean': float(np.mean(loss_gaps)), 'sd': float(np.std(loss_gaps, ddof=1))},
        'val_rmse': per_horizon_rmse,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved summary to {SUMMARY_PATH}')

=== Diagnostic 3: partial freeze, shared head (resnet18, real images) ===
Best epoch: 7.30 +/- 2.21; stopping epoch: 17.30 +/- 2.21
Final (val - train) loss gap: -0.015051 +/- 0.003710
 15-min:  RMSE = 10.56 +/- 0.04
 30-min:  RMSE = 16.60 +/- 0.06
 60-min:  RMSE = 23.92 +/- 0.07
 90-min:  RMSE = 28.37 +/- 0.08
120-min:  RMSE = 31.22 +/- 0.08

saved per-seed results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_3_resnet
saved summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_3_resnet/diagnostic3_partial_freeze_shared_head_resnet18_summary.json


In [21]:
# Paired multi-seed VALIDATION analysis
#
# This cell compares the same 10 seeds across:
#   1) CGM+Time baseline
#   2) Fully joint multimodal model at img_proj_dim=4
#   3) Diagnostic 3 partial-freeze model at img_proj_dim=4
#
# All comparisons are descriptive and paired by seed. No held-out test results are used.

BASE_DIR = (
    PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
    / 'results' / 'gated_cgm_time'
)
DIAG2_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_2_resnet'


def _get_horizon_entry(results_dict, horizon):
    """Handle JSON dictionaries whose horizon keys may be strings or integers."""
    if str(horizon) in results_dict:
        return results_dict[str(horizon)]
    if horizon in results_dict:
        return results_dict[horizon]
    raise KeyError(f'Horizon {horizon} not found. Available keys: {list(results_dict.keys())}')


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    """Load one result JSON per seed and return {seed: full_result_dict}."""
    loaded = {}

    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'

        if not path.exists():
            raise FileNotFoundError(
                f'Missing result file for seed {seed}: {path}\n'
                f'Check result_dir/stub before running this comparison.'
            )

        with open(path) as f:
            result = json.load(f)

        if result_key not in result:
            raise KeyError(
                f'{path.name} does not contain "{result_key}". '
                f'Available keys: {list(result.keys())}'
            )

        loaded[seed] = result

    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='val_results'):
    """Return {horizon: np.array([RMSE for each seed in SEEDS order])}."""
    out = {}

    for h in horizons:
        vals = []
        for seed in seeds:
            entry = _get_horizon_entry(loaded_results[seed][result_key], h)
            vals.append(float(entry['rmse']))
        out[h] = np.asarray(vals, dtype=float)

    return out


# Load the two previous experiments from disk.
base_loaded = load_seed_results(
    BASE_DIR,
    'gated_cgm_time',
    SEEDS,
    result_key='val_results',
)

joint_loaded = load_seed_results(
    DIAG2_DIR,
    'diagnostic2_proj4',
    SEEDS,
    result_key='val_results',
)

base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)
joint_by_h = rmse_by_seed(joint_loaded, SEEDS, HORIZONS)

# Diagnostic 3 results are already in memory from the current notebook run.
partial_by_h = {
    h: np.asarray(
        [float(_get_horizon_entry(r['val_results'], h)['rmse']) for r in all_results],
        dtype=float,
    )
    for h in HORIZONS
}

# Safety check: current in-memory results must correspond exactly to SEEDS.
current_seeds = [int(r['seed']) for r in all_results]
if current_seeds != SEEDS:
    raise ValueError(
        f'Current Diagnostic 3 seed order does not match SEEDS.\n'
        f'Expected: {SEEDS}\n'
        f'Found:    {current_seeds}'
    )


print('=== Paired validation RMSE across seeds ===')
print(
    f"{'horizon':>9}"
    f"{'CGM+time':>19}"
    f"{'fully joint d=4':>22}"
    f"{'partial freeze d=4':>24}"
)
for h in HORIZONS:
    base = base_by_h[h]
    joint = joint_by_h[h]
    partial = partial_by_h[h]

    print(
        f"{h:>6} min"
        f"{np.mean(base):>10.3f} +/- {np.std(base, ddof=1):<6.3f}"
        f"{np.mean(joint):>13.3f} +/- {np.std(joint, ddof=1):<6.3f}"
        f"{np.mean(partial):>15.3f} +/- {np.std(partial, ddof=1):<6.3f}"
    )


print('\n=== Diagnostic 3 paired effect: Partial Freeze vs CGM+Time ===')
print(
    f"{'horizon':>9}"
    f"{'delta RMSE':>22}"
    f"{'improvement %':>24}"
    f"{'improved seeds':>18}"
)
for h in HORIZONS:
    base = base_by_h[h]
    partial = partial_by_h[h]

    # Negative delta = lower RMSE = better.
    delta = partial - base

    # Positive percentage = improvement.
    improvement_pct = (base - partial) / base * 100.0

    wins = int(np.sum(partial < base))
    ties = int(np.sum(np.isclose(partial, base, atol=1e-12)))
    losses = len(SEEDS) - wins - ties

    print(
        f"{h:>6} min"
        f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
        f"{np.mean(improvement_pct):>13.3f} +/- {np.std(improvement_pct, ddof=1):<7.3f}"
        f"{wins:>8}/{len(SEEDS)}"
        f"  (ties={ties}, losses={losses})"
    )


print('\n=== Effect of freezing: Partial Freeze vs Fully Joint (both d=4) ===')
print(
    f"{'horizon':>9}"
    f"{'delta RMSE':>22}"
    f"{'improvement %':>24}"
    f"{'PF better seeds':>18}"
)
for h in HORIZONS:
    joint = joint_by_h[h]
    partial = partial_by_h[h]

    # Negative delta means Partial Freeze has lower RMSE.
    delta = partial - joint
    improvement_pct = (joint - partial) / joint * 100.0

    wins = int(np.sum(partial < joint))
    ties = int(np.sum(np.isclose(partial, joint, atol=1e-12)))
    losses = len(SEEDS) - wins - ties

    print(
        f"{h:>6} min"
        f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
        f"{np.mean(improvement_pct):>13.3f} +/- {np.std(improvement_pct, ddof=1):<7.3f}"
        f"{wins:>8}/{len(SEEDS)}"
        f"  (ties={ties}, losses={losses})"
    )


# Optional seed-level detail table. This is useful for checking whether the mean
# improvement is broad across seeds or driven by only one or two runs.
rows = []

for h in HORIZONS:
    for i, seed in enumerate(SEEDS):
        base = base_by_h[h][i]
        joint = joint_by_h[h][i]
        partial = partial_by_h[h][i]

        rows.append({
            'horizon_min': h,
            'seed': seed,
            'cgm_time_rmse': base,
            'fully_joint_dim4_rmse': joint,
            'partial_freeze_dim4_rmse': partial,
            'partial_minus_base': partial - base,
            'partial_minus_joint': partial - joint,
            'partial_improvement_vs_base_pct': (base - partial) / base * 100.0,
        })

# paired_validation_df = pd.DataFrame(rows)

# print('\n=== Per-seed paired validation detail ===')
# display(paired_validation_df.round(4))


=== Paired validation RMSE across seeds ===
  horizon           CGM+time       fully joint d=4      partial freeze d=4
    15 min    10.580 +/- 0.073        10.660 +/- 0.092          10.560 +/- 0.035 
    30 min    16.705 +/- 0.060        16.826 +/- 0.085          16.601 +/- 0.064 
    60 min    24.087 +/- 0.063        24.274 +/- 0.121          23.916 +/- 0.068 
    90 min    28.415 +/- 0.047        28.792 +/- 0.139          28.374 +/- 0.075 
   120 min    31.270 +/- 0.050        31.624 +/- 0.161          31.221 +/- 0.083 

=== Diagnostic 3 paired effect: Partial Freeze vs CGM+Time ===
  horizon            delta RMSE           improvement %    improved seeds
    15 min    -0.0204 +/- 0.0803         0.188 +/- 0.753         5/10  (ties=0, losses=5)
    30 min    -0.1035 +/- 0.0641         0.619 +/- 0.383         9/10  (ties=0, losses=1)
    60 min    -0.1703 +/- 0.0566         0.707 +/- 0.235        10/10  (ties=0, losses=0)
    90 min    -0.0412 +/- 0.0671         0.145 +/- 0.236       